This notebook helps parse Quantrocket API specs into MCP resources for the strategy coder agent to use.

In [6]:
from bs4 import BeautifulSoup
from pathlib import Path

html = Path("zipline_api.html").read_text()
soup = BeautifulSoup(html, "html.parser")

In [4]:
def get_category_path(section):
    path = []
    parent = section
    while parent:
        header = parent.find(["h1", "h2"])
        if header:
            path.append(header.get_text(strip=True))
        parent = parent.find_parent("section")
    return list(reversed(path))

def parse_class(dl, category_path):
    sig = dl.find("dt", class_="sig")
    class_name = sig.find("span", class_="sig-name").text

    full_name = "zipline.api." + class_name

    description = " ".join(
        p.get_text(" ", strip=True)
        for p in dl.find_all("p", recursive=False)
    )

    return {
        "id": full_name,
        "kind": "class",
        "category": category_path,
        "signature": sig.get_text(" ", strip=True),
        "summary": description,
        "scope": ["initialize", "handle_data", "before_trading_start"],
        "methods": []
    }

SCOPE_OVERRIDES = {
    "schedule_function": ["initialize"],
    "pipeline_output": ["before_trading_start", "scheduled_function"],
    "order": ["handle_data", "scheduled_function"],
}

def infer_scope(symbol_id):
    name = symbol_id.split(".")[-1]
    return SCOPE_OVERRIDES.get(
        name,
        ["handle_data", "before_trading_start", "scheduled_function"]
    )

def parse_method(dl, class_id, category_path):
    sig = dl.find("dt", class_="sig")
    name = sig.find("span", class_="sig-name").text
    full_id = f"{class_id}.{name}"

    summary = dl.find("p").get_text(" ", strip=True)

    params = []
    returns = None

    field_list = dl.find("dl", class_="field-list")
    if field_list:
        for dt in field_list.find_all("dt"):
            label = dt.get_text(strip=True)
            dd = dt.find_next_sibling("dd")

            if label.startswith("Parameters"):
                for li in dd.find_all("li"):
                    params.append(li.get_text(" ", strip=True))

            elif label.startswith("Returns"):
                returns = dd.get_text(" ", strip=True)

    rules = [
        li.get_text(" ", strip=True)
        for li in dl.find_all("ul", class_="simple")
    ]

    return {
        "id": full_id,
        "kind": "method",
        "owner": class_id,
        "category": category_path,
        "signature": sig.get_text(" ", strip=True),
        "summary": summary,
        "parameters": params,
        "returns": returns,
        "rules": rules,
        "scope": infer_scope(full_id)
    }

In [ ]:
results = {}

for section in soup.find_all("section"):
    # Check if section starts with h2 block
    if section.find("h2", recursive=False):
        # Add section id as new key in results
        results[section['id']] = {}
    elif section.find("h3", recursive=False):
        # Add section as new key in latest section in results
        parent_section = section.find_parent("section")
        if parent_section and parent_section['id'] in results:
            # Add subsection id as new key in parent section and remainder of the h3 section as value
            results[parent_section['id']][section['id']] = "temp"
    else:
        continue

print(results)

{'data-object': {'zipline-api-bardata': 'temp'}, 'context-object': {'zipline-api-context': 'temp'}, 'scheduling-functions': {'zipline-api-schedule-function': 'temp', 'zipline-api-date-rules': 'temp', 'zipline-api-time-rules': 'temp'}, 'simulation-date': {'zipline-api-get-datetime': 'temp'}, 'ordering': {'zipline-api-order': 'temp', 'zipline-api-order-value': 'temp', 'zipline-api-order-percent': 'temp', 'zipline-api-order-target': 'temp', 'zipline-api-order-target-value': 'temp', 'zipline-api-order-target-percent': 'temp'}, 'execution-styles': {'zipline-finance-execution-marketorder': 'temp', 'zipline-finance-execution-limitorder': 'temp', 'zipline-finance-execution-stoporder': 'temp', 'zipline-finance-execution-stoplimitorder': 'temp', 'zipline-finance-execution-marketoncloseorder': 'temp', 'zipline-finance-execution-limitoncloseorder': 'temp', 'zipline-finance-execution-marketonopenorder': 'temp', 'zipline-finance-execution-limitonopenorder': 'temp'}, 'order-management': {'zipline-api